# DSAN 6600 - 03_cnn_baseline.ipynb

Neural baseline for Check-in 2: a 1D residual CNN over the raw 12-lead ECG (100 Hz), multi-label over the 5 diagnostic superclasses.

The **same architecture** is trained three ways, changing only who is in the training data:

| Model | Trains on | Validates on |
|---|---|---|
| `cnn_female` | female records, folds 1–7 | female records, fold 8 |
| `cnn_male` | male records, folds 1–7 | male records, fold 8 |
| `cnn_all` | all records, folds 1–7 | all records, fold 8 |

Every model is then tested on **both** the male and the female test sets (folds 9–10), giving a 3 × 2 grid.

**Running on Colab (recommended, GPU):**
1. Open this notebook in Colab (File → Open notebook → GitHub → `chloemjones13/6600_final_project`).
2. Runtime → Change runtime type → **T4 GPU**.
3. Runtime → Run all. The setup cells mount Google Drive, clone the repo, build `ptbxl_model.pkl` by running `01_data_cleaning.ipynb`, and download PTB-XL (first session only).
4. After **each** training run, its predictions, history and checkpoint are copied to `MyDrive/6600_project/backup/`. If Colab disconnects, just Run all again: finished runs are restored from Drive and skipped.
5. The last cell zips `results/` and `figures/` for download so they can be committed to the repo.

**Running locally:** run `01_data_cleaning.ipynb` first, and have the PTB-XL 100 Hz files in `data/` (see README for the `wget`). Drive steps are skipped automatically.

**Outputs** (format defined in `EVAL.md`):
- `results/cnn_{female,male,all}_seed{0,1,2}_preds.csv` — validation + test probabilities (both sexes), one row per record
- `results/cnn_{group}_seed{seed}_history.csv`, `results/cnn_config.json` — training logs and setup
- `figures/cnn_training_curves.png`, `figures/cnn_examples_*.png`
- `checkpoints/cnn_{group}_seed{seed}.pt` — best weights (gitignored)

### Training setup

| | |
|---|---|
| **Input** | 12 leads × 1,000 samples (10 s at 100 Hz), per-lead z-score using mean/std from **that model's own training records** only |
| **Splits** | train = folds 1–7, validation = fold 8, test = folds 9–10 (patients never cross folds); train and validation filtered to the model's group, test always includes both sexes |
| **Model** | ResNet-style 1D CNN: stem conv (k=15, stride 2) + max-pool → 4 residual blocks (64→64→128→128→256 channels, k=7, two downsampling) → global average pool → dropout → linear (5) |
| **Output** | 5 independent sigmoids (multi-label) |
| **Loss** | `BCEWithLogitsLoss`, unweighted |
| **Optimizer** | AdamW, lr 1e-3, weight decay 1e-4, batch 64 |
| **Schedule** | ReduceLROnPlateau on validation macro AUROC (factor 0.5, patience 3) |
| **Stopping** | max 40 epochs, early stop after 7 epochs without validation improvement; keep the best-validation-AUROC checkpoint |
| **Seeds** | 0, 1, 2 per model (each changes weight init and batch order) → 9 training runs |

**Why no class weights:** a positive-class weight raises predicted probabilities for rare classes, which distorts calibration, and calibration by sex is one of our metrics. AUROC doesn't depend on thresholds, so imbalance doesn't need correcting for it.

**Model architecture is fixed** for the whole project; only the training data changes.

**Training sizes differ:** about 6,600 female vs 7,400 male vs 14,000 combined records. So `cnn_all` vs the single-sex models compares *more data* and *mixed data* at once. Set `MATCH_SIZE = True` to subsample every group to the size of the smallest one (the female training set), which isolates composition from size. This is the start of RQ1/RQ2.

### Setup (Colab + paths)

In [ ]:
import os
import sys
import glob
import shutil
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/chloemjones13/6600_final_project.git"
REPO_DIR = "/content/6600_final_project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/6600_project/"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR + "/notebooks")
    CACHE_DIR = DRIVE_DIR             # signal cache lives on Drive so it survives sessions
    BACKUP_DIR = DRIVE_DIR + "backup/"
else:
    CACHE_DIR = "../data/"
    BACKUP_DIR = None

PATH = "../data/"                     # ptbxl_model.pkl
SIGNAL_DIR = "../data/"               # folder containing records100/ (Colab: set below)
RESULTS_DIR, CKPT_DIR, FIG_DIR = "../results/", "../checkpoints/", "../figures/"
for d in [RESULTS_DIR, CKPT_DIR, FIG_DIR, CACHE_DIR] + ([BACKUP_DIR] if BACKUP_DIR else []):
    os.makedirs(d, exist_ok=True)

# Build the cleaned modeling data if it's missing (runs notebook 01; needs only the repo CSVs)
if not os.path.exists(PATH + "ptbxl_model.pkl"):
    subprocess.run(["jupyter", "nbconvert", "--to", "notebook", "--execute",
                    "--output", "/tmp/01_executed.ipynb", "01_data_cleaning.ipynb"], check=True)
assert os.path.exists(PATH + "ptbxl_model.pkl"), "01_data_cleaning.ipynb didn't create ptbxl_model.pkl"

print("Colab:", IN_COLAB, "| working dir:", os.getcwd())

In [ ]:
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

def save(name):
    plt.savefig(f"{FIG_DIR}{name}.png", dpi=150, bbox_inches="tight")
    if BACKUP_DIR:
        os.makedirs(BACKUP_DIR + "figures", exist_ok=True)
        shutil.copy(f"{FIG_DIR}{name}.png", BACKUP_DIR + "figures/")

CLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("Device:", DEVICE)
if IN_COLAB and DEVICE.type != "cuda":
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then Run all again.")

In [ ]:
CONFIG = dict(
    model="cnn",
    architecture="ResNet1d: stem conv + 4 residual blocks (9 conv layers), global avg pool, linear head",
    sampling_rate_hz=100,
    input_shape=[12, 1000],
    train_folds=[1, 2, 3, 4, 5, 6, 7],
    val_folds=[8],
    test_folds=[9, 10],
    normalization="per-lead z-score, mean/std from each model's own training records",
    loss="BCEWithLogitsLoss (unweighted)",
    optimizer="AdamW",
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    max_epochs=40,
    early_stopping_patience=7,
    scheduler="ReduceLROnPlateau(mode=max on val macro AUROC, factor=0.5, patience=3)",
    dropout=0.2,
    selection_metric="validation macro AUROC",
    seeds=[0, 1, 2],
    train_groups=["female", "male", "all"],
    match_size=False,
)
MATCH_SIZE = CONFIG["match_size"]   # True = subsample every group to the smallest group's size

### Load the cleaned data and define splits

In [ ]:
df = pd.read_pickle(PATH + "ptbxl_model.pkl")
assert (df[CLASSES].sum(axis=1) > 0).all(), "Run the new last cell of 01_data_cleaning.ipynb first"

df["split"] = np.where(df["strat_fold"] <= 7, "train", np.where(df["strat_fold"] == 8, "val", "test"))
print(pd.crosstab(df["split"], df["sex_label"]))

**Check:** these counts should match Olivia's exactly (expected test: 2,137 male / 1,842 female).

### Load signals

Reading ~20,000 files with `wfdb` takes a few minutes, so the first run caches the array (~1 GB) to `X_100hz.npy`: in `data/` locally (gitignored), or on Google Drive in Colab. Later runs and later Colab sessions load the cache and skip the download.

In [ ]:
CACHE = CACHE_DIR + "X_100hz.npy"
CACHE_IDS = CACHE_DIR + "X_100hz_ids.npy"

# Colab, first session only: download PTB-XL (1.7 GB) and extract just the 100 Hz records
if IN_COLAB and not os.path.exists(CACHE):
    if not glob.glob("/content/ptbxl/**/records100", recursive=True):
        subprocess.run(["wget", "-q", "-O", "/content/ptbxl.zip",
                        "https://physionet.org/content/ptb-xl/get-zip/1.0.3/"], check=True)
        subprocess.run(["unzip", "-q", "/content/ptbxl.zip", "*records100/*", "-d", "/content/ptbxl"], check=True)
    SIGNAL_DIR = os.path.dirname(glob.glob("/content/ptbxl/**/records100", recursive=True)[0]) + "/"

def load_signals(df):
    if os.path.exists(CACHE) and os.path.exists(CACHE_IDS):
        if np.array_equal(np.load(CACHE_IDS), df.index.values):
            return np.load(CACHE)
        print("Cache doesn't match the current data; rebuilding")
    X = np.empty((len(df), 12, 1000), dtype=np.float32)
    for i, fn in enumerate(df["filename_lr"]):
        sig, _ = wfdb.rdsamp(SIGNAL_DIR + fn)
        X[i] = sig.T
        if i % 2000 == 0:
            print(f"{i}/{len(df)}")
    np.save(CACHE, X)
    np.save(CACHE_IDS, df.index.values)
    return X

X = load_signals(df)
print(X.shape)
assert np.isfinite(X).all()

### Split by fold and sex

Signals stay in raw units here. Each model normalizes with its own training records' statistics (stored inside the model, see below), so the female-only model never sees anything computed from male data.

In [ ]:
SPLITS = ["train", "val", "test"]
masks = {s: (df["split"] == s).values for s in SPLITS}

Xs = {s: X[masks[s]] for s in SPLITS}
del X
Ys = {s: df.loc[masks[s], CLASSES].values.astype(np.float32) for s in SPLITS}
meta = {s: df[masks[s]] for s in SPLITS}

def group_mask(split, group):
    sex = meta[split]["sex"].values
    if group == "all":
        return np.ones(len(sex), bool)
    return sex == (1 if group == "female" else 0)

for g in CONFIG["train_groups"]:
    print(f"{g:6s} train {group_mask('train', g).sum():6d} | val {group_mask('val', g).sum():5d}")

### Model

In [ ]:
class ResBlock1d(nn.Module):
    def __init__(self, c_in, c_out, stride=1, kernel=7, dropout=0.2):
        super().__init__()
        pad = kernel // 2
        self.conv1 = nn.Conv1d(c_in, c_out, kernel, stride=stride, padding=pad, bias=False)
        self.bn1 = nn.BatchNorm1d(c_out)
        self.conv2 = nn.Conv1d(c_out, c_out, kernel, padding=pad, bias=False)
        self.bn2 = nn.BatchNorm1d(c_out)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(dropout)
        if stride != 1 or c_in != c_out:
            self.shortcut = nn.Sequential(
                nn.Conv1d(c_in, c_out, 1, stride=stride, bias=False),
                nn.BatchNorm1d(c_out),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = self.drop(self.relu(self.bn1(self.conv1(x))))
        out = self.bn2(self.conv2(out))
        return self.relu(out + self.shortcut(x))


class ECGResNet1d(nn.Module):
    def __init__(self, n_leads=12, n_classes=5, dropout=0.2):
        super().__init__()
        # per-lead normalization stats, set from the training records before training
        self.register_buffer("mean", torch.zeros(1, n_leads, 1))
        self.register_buffer("std", torch.ones(1, n_leads, 1))
        self.stem = nn.Sequential(
            nn.Conv1d(n_leads, 64, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1),
        )
        self.blocks = nn.Sequential(
            ResBlock1d(64, 64, stride=1, dropout=dropout),
            ResBlock1d(64, 128, stride=2, dropout=dropout),
            ResBlock1d(128, 128, stride=1, dropout=dropout),
            ResBlock1d(128, 256, stride=2, dropout=dropout),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256, n_classes),
        )

    def forward(self, x):          # x: (batch, 12, 1000), raw millivolts
        x = (x - self.mean) / self.std
        return self.head(self.blocks(self.stem(x)))   # logits: (batch, 5)


m = ECGResNet1d()
print("Parameters:", sum(p.numel() for p in m.parameters()))
print("Output shape:", m(torch.zeros(2, 12, 1000)).shape)   # expect (2, 5)
del m

### Training functions

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def make_loader(X, Y, shuffle, seed=0):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(Y))
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(ds, batch_size=CONFIG["batch_size"], shuffle=shuffle,
                      generator=g if shuffle else None, num_workers=0)


@torch.no_grad()
def predict(model, loader):
    model.eval()
    out = []
    for xb, _ in loader:
        out.append(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
    return np.concatenate(out)


def bce(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))


SMALLEST = min(group_mask("train", g).sum() for g in CONFIG["train_groups"])

def train_one(group, seed):
    set_seed(seed)
    tr = np.where(group_mask("train", group))[0]
    if MATCH_SIZE and len(tr) > SMALLEST:
        tr = np.sort(np.random.default_rng(seed).choice(tr, SMALLEST, replace=False))
    va = group_mask("val", group)
    X_tr, Y_tr = Xs["train"][tr], Ys["train"][tr]
    X_va, Y_va = Xs["val"][va], Ys["val"][va]

    model = ECGResNet1d(dropout=CONFIG["dropout"])
    model.mean.copy_(torch.from_numpy(X_tr.mean(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)))
    model.std.copy_(torch.from_numpy(X_tr.std(axis=(0, 2), keepdims=True, dtype=np.float64).astype(np.float32)))
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    loss_fn = nn.BCEWithLogitsLoss()
    train_loader = make_loader(X_tr, Y_tr, shuffle=True, seed=seed)
    val_loader = make_loader(X_va, Y_va, shuffle=False)
    print(f"--- cnn_{group}, seed {seed}: {len(X_tr)} train / {len(X_va)} val records")

    ckpt = f"{CKPT_DIR}cnn_{group}_seed{seed}.pt"
    best, bad, hist = -1.0, 0, []
    for epoch in range(1, CONFIG["max_epochs"] + 1):
        t0 = time.time()
        model.train()
        total, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
            n += len(xb)

        p_val = predict(model, val_loader)
        val_auc = roc_auc_score(Y_va, p_val, average="macro")
        sched.step(val_auc)
        hist.append(dict(epoch=epoch, train_loss=total / n, val_loss=bce(Y_va, p_val),
                         val_auroc=val_auc, lr=opt.param_groups[0]["lr"], sec=time.time() - t0))
        h = hist[-1]
        print(f"{group} seed {seed} | epoch {epoch:2d} | train loss {h['train_loss']:.4f} | "
              f"val loss {h['val_loss']:.4f} | val AUROC {val_auc:.4f} | {h['sec']:.0f}s")

        if val_auc > best:
            best, bad = val_auc, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= CONFIG["early_stopping_patience"]:
                print(f"Early stop at epoch {epoch}; best val AUROC {best:.4f}")
                break

    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    hist = pd.DataFrame(hist)
    hist["n_train"] = len(X_tr)
    return model, hist


def save_preds(model, group, seed, n_train):
    """Validation + test predictions (both sexes) in the EVAL.md format."""
    parts = []
    for s in ["val", "test"]:
        p = predict(model, make_loader(Xs[s], Ys[s], shuffle=False))
        m = meta[s]
        out = pd.DataFrame({
            "ecg_id": m.index.values,
            "split": s,
            "sex": m["sex"].values,
            "age": m["age"].values,
            "age_band": m["age_band"].astype(str).values,
            "device": m["device"].values,
            "validated_by_human": m["validated_by_human"].values,
        })
        for j, c in enumerate(CLASSES):
            out["y_" + c] = Ys[s][:, j].astype(int)
        for j, c in enumerate(CLASSES):
            out["p_" + c] = p[:, j]
        out["model"] = f"cnn_{group}"
        out["train_group"] = group
        out["n_train"] = n_train
        out["seed"] = seed
        parts.append(out)
    pd.concat(parts).to_csv(f"{RESULTS_DIR}cnn_{group}_seed{seed}_preds.csv", index=False)

### Train (3 models × 3 seeds = 9 runs)

On a Colab T4 GPU expect a few seconds per epoch, so all 9 runs should take well under an hour. On a laptop CPU it's roughly 30–60 s per epoch for the combined model, so a couple of hours. Runs usually early-stop well before 40 epochs. Finished runs are skipped if you rerun, so an interruption doesn't lose work.

In [ ]:
def run_files(group, seed):
    name = f"cnn_{group}_seed{seed}"
    return [f"{RESULTS_DIR}{name}_preds.csv", f"{RESULTS_DIR}{name}_history.csv", f"{CKPT_DIR}{name}.pt"]

def backup(group, seed):
    if BACKUP_DIR:
        for f in run_files(group, seed):
            shutil.copy(f, BACKUP_DIR)
        print(f"Backed up cnn_{group} seed {seed} to Drive")

def restore(group, seed):
    """Copy a finished run back from Drive (new Colab session). Returns True if all files are present."""
    for f in run_files(group, seed):
        b = BACKUP_DIR + os.path.basename(f) if BACKUP_DIR else None
        if not os.path.exists(f) and b and os.path.exists(b):
            shutil.copy(b, f)
    return all(os.path.exists(f) for f in run_files(group, seed))

histories = {}
for group in CONFIG["train_groups"]:
    for seed in CONFIG["seeds"]:
        pred_file, hist_file, _ = run_files(group, seed)
        if restore(group, seed):
            print(f"cnn_{group} seed {seed}: already done, skipping")
            histories[(group, seed)] = pd.read_csv(hist_file)
            continue
        model, hist = train_one(group, seed)
        histories[(group, seed)] = hist
        hist.to_csv(hist_file, index=False)
        save_preds(model, group, seed, int(hist["n_train"].iloc[0]))
        backup(group, seed)

with open(RESULTS_DIR + "cnn_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
if BACKUP_DIR:
    shutil.copy(RESULTS_DIR + "cnn_config.json", BACKUP_DIR)

In [ ]:
groups = CONFIG["train_groups"]
fig, axes = plt.subplots(2, len(groups), figsize=(5 * len(groups), 7), sharex=True)
for j, g in enumerate(groups):
    for seed in CONFIG["seeds"]:
        h = histories[(g, seed)]
        axes[0, j].plot(h["epoch"], h["train_loss"], label=f"train, seed {seed}")
        axes[0, j].plot(h["epoch"], h["val_loss"], "--", label=f"val, seed {seed}")
        axes[1, j].plot(h["epoch"], h["val_auroc"], label=f"seed {seed}")
    axes[0, j].set_title(f"cnn_{g}: loss (BCE)")
    axes[1, j].set_title(f"cnn_{g}: validation macro AUROC")
    axes[1, j].set_xlabel("epoch")
axes[0, 0].legend(fontsize=7); axes[1, 0].legend(fontsize=7)
plt.tight_layout(); save("cnn_training_curves"); plt.show()

### Quick test-set check

A sanity check only. The full evaluation (calibration, slices, comparison with logistic regression) runs in the shared eval notebook following `EVAL.md`.

In [ ]:
Y_COLS = ["y_" + c for c in CLASSES]
P_COLS = ["p_" + c for c in CLASSES]

def summarize(preds):
    t = preds[preds["split"] == "test"]
    r = {}
    for name, mask in [("all", np.ones(len(t), bool)), ("male", (t["sex"] == 0).values), ("female", (t["sex"] == 1).values)]:
        y, p = t.loc[mask, Y_COLS].values, t.loc[mask, P_COLS].values
        r[f"auroc_{name}"] = roc_auc_score(y, p, average="macro")
        r[f"auprc_{name}"] = average_precision_score(y, p, average="macro")
    r["gap_auroc (M-F)"] = r["auroc_male"] - r["auroc_female"]
    return r

preds = {(g, s): pd.read_csv(f"{RESULTS_DIR}cnn_{g}_seed{s}_preds.csv")
         for g in CONFIG["train_groups"] for s in CONFIG["seeds"]}
res = pd.DataFrame([dict(model=f"cnn_{g}", seed=s, **summarize(p)) for (g, s), p in preds.items()])
summary = res.drop(columns="seed").groupby("model", sort=False).agg(["mean", "std"]).round(3)
summary

In [ ]:
# The 3 x 2 grid: macro AUROC by training group (rows) and test sex (columns), mean over seeds
grid = summary[[("auroc_male", "mean"), ("auroc_female", "mean")]]
grid.columns = ["test: male", "test: female"]
grid

In [ ]:
# Per-class test AUROC: training group x test sex, mean over seeds
rows = []
for (g, s), pr in preds.items():
    t = pr[pr["split"] == "test"]
    for sex, name in [(0, "male"), (1, "female")]:
        tt = t[t["sex"] == sex]
        for c in CLASSES:
            rows.append(dict(model=f"cnn_{g}", test=name, cls=c, auroc=roc_auc_score(tt["y_" + c], tt["p_" + c])))
per_class = pd.DataFrame(rows).groupby(["model", "cls", "test"], sort=False)["auroc"].mean().unstack()[["male", "female"]]
per_class["gap (M-F)"] = per_class["male"] - per_class["female"]
per_class.round(3)

### Qualitative examples (seed 0)

Shown for one model at a time (`EXAMPLE_MODEL`); compare `all` vs `female` vs `male` on the same classes.

For each class: a confident correct positive, the worst miss (a true positive with the lowest predicted probability), and the worst false alarm (a negative with the highest predicted probability). Lead II and V5 shown in raw millivolts (from the loaded signals, so no extra files are needed). Change `EXAMPLE_CLASSES` to look at other classes.

In [ ]:
EXAMPLE_MODEL = "all"            # "all", "female" or "male"
EXAMPLE_CLASSES = ["MI", "STTC", "HYP"]

p0 = preds[(EXAMPLE_MODEL, 0)]
t0 = p0[p0["split"] == "test"].set_index("ecg_id")
t0 = t0.join(df[["labels"]])
test_row = {eid: i for i, eid in enumerate(meta["test"].index)}   # ecg_id -> row in Xs["test"]
time_axis = np.arange(1000) / 100

def show_examples(c):
    pos, neg = t0[t0["y_" + c] == 1], t0[t0["y_" + c] == 0]
    picks = [("Correct (confident)", pos["p_" + c].idxmax()),
             ("Missed (false negative)", pos["p_" + c].idxmin()),
             ("False alarm (false positive)", neg["p_" + c].idxmax())]
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.2), sharey=True)
    for ax, (kind, eid) in zip(axes, picks):
        r = t0.loc[eid]
        sig = Xs["test"][test_row[eid]]          # (12, 1000), raw mV
        ax.plot(time_axis, sig[1], lw=0.8, label="II")
        ax.plot(time_axis, sig[10] - 2, lw=0.8, label="V5 (shifted down)")
        sex = "female" if r["sex"] == 1 else "male"
        ax.set_title(f"cnn_{EXAMPLE_MODEL} | {c} | {kind}\nid {eid}, {sex}, age {int(r['age'])}, {r['device']}\n"
                     f"true: {', '.join(r['labels'])} | p({c}) = {r['p_' + c]:.2f}", fontsize=9)
        ax.set_xlabel("seconds")
    axes[0].legend(fontsize=8)
    plt.tight_layout(); save(f"cnn_{EXAMPLE_MODEL}_examples_{c}"); plt.show()

for c in EXAMPLE_CLASSES:
    show_examples(c)

In [ ]:
# Where do errors concentrate? Share of false negatives / false positives at a 0.5 cutoff, by sex
# (0.5 is only for counting errors here; the reported metrics don't use a threshold)
rows = []
for c in CLASSES:
    for sex, name in [(0, "male"), (1, "female")]:
        tt = t0[t0["sex"] == sex]
        pos, neg = tt[tt["y_" + c] == 1], tt[tt["y_" + c] == 0]
        rows.append(dict(cls=c, sex=name, n_pos=len(pos),
                         miss_rate=(pos["p_" + c] < 0.5).mean(),
                         false_alarm_rate=(neg["p_" + c] >= 0.5).mean()))
pd.DataFrame(rows).set_index(["cls", "sex"]).round(3)

### Notes for the failure analysis

Fill in after running:
- Does the female-only model beat the combined model on women, or does the combined model's extra data win?
- Does each single-sex model drop when tested on the other sex? By how much?
- Which classes have the largest male − female gap?
- Do the misses look like the EDA predicted (lower chest-lead voltage in women hiding HYP; ST/T differences affecting MI and STTC)?
- Is the gap still there within a single device (check CS100 vs others in the eval notebook)?

### Download outputs (Colab)

Zips `results/` and `figures/` so they can be added to the repo. A copy is also saved to Drive. Checkpoints stay on Drive only (they're gitignored).

In [ ]:
if IN_COLAB:
    subprocess.run(["zip", "-qr", "/content/cnn_outputs.zip", "results", "figures"], cwd="..", check=True)
    shutil.copy("/content/cnn_outputs.zip", DRIVE_DIR)
    from google.colab import files
    files.download("/content/cnn_outputs.zip")